# Introduction to Cheminformatics with RDKit

In this notebook we explore how molecules can be represented, compared, and analyzed using RDKit.

We will move progressively from simple textual representations to full 3D structures:

- Representing molecules using SMILES
- Comparing molecules using fingerprints and similarity
- Generating 3D structures

The goal is to understand how chemistry can be translated into:

- Graphs
- Mathematical descriptors
- Spatial geometry
- Computational workflows

Let's start by importing the required libraries and setting up our environment.

In [ ]:
# Core scientific computing
import pandas as pd

try:
    from IPython.display import display
except:
    !pip install IPython
    from IPython.display import display

# RDKit imports
try:
    from rdkit import Chem
    from rdkit.Chem import (
        Draw, AllChem, Descriptors
    )
except:
    !pip install rdkit # pip install --force-reinstall -v rdkit==2025.03.6
    from rdkit import Chem
    from rdkit.Chem import (
        Draw, AllChem, Descriptors
    )

# 3D visualization
try:
    import py3Dmol
except:
    !pip install py3Dmol
    import py3Dmol

# Understanding SMILES Notation

SMILES (Simplified Molecular Input Line Entry System) is a line notation for describing the structure of chemical species using short ASCII strings. For example:
- `CCC` represents propane
- `C=C` represents ethene
- `c1ccccc1` represents benzene (aromatic)
- `CC(=O)O` represents acetic acid

It tells us:
- Which atoms are present
- How they are connected

It does NOT contain:
- 3D coordinates
- Explicit geometry

#### What is `Mol` in RDKit?

In RDKit, a `Mol` object is not just a "molecule" in the traditional chemical sense.

It is a **graph representation** of a molecule.

When we write:

```python
mol = Chem.MolFromSmiles("CCC")
```

we are creating a graph where:

- Atoms are nodes  
- Bonds are edges  

Internally, RDKit stores:

- 3 atoms  
- 2 bonds  
- Atom indices (0, 1, 2)  
- Bond types (single bonds)  
- Connectivity information (which atoms are connected)  

At this stage, the molecule contains:

- No 3D coordinates  
- No energy information  
- No conformers  

It is purely a connectivity graph.

Only after generating conformers does a `Mol` object contain 3D geometry.


In [ ]:
# How to use rdkit molecule objects
smiles_1 = 'CC=C' # SMILES string for propene
mol = Chem.MolFromSmiles(smiles_1)

# We need to add Hydrogens to the molecule to get the correct number of atoms
mol = Chem.AddHs(mol)

display(mol)


In [ ]:
# Loop over all atoms in the molecule
for atom in mol.GetAtoms():
    
    # Get the atom index (its position in the molecule)
    idx = atom.GetIdx()
    
    # Get the chemical symbol (C, O, N, etc.)
    symbol = atom.GetSymbol()
    
    # Get the degree (number of directly bonded atoms)
    degree = atom.GetDegree()
    
    # Print the information
    print(f"Atom {idx}: {symbol} (degree = {degree})")

In [ ]:
# Get the atom with index 1 from the molecule
atom = mol.GetAtomWithIdx(1)

# Print basic information about this atom
print(f"Atom {atom.GetIdx()}: {atom.GetSymbol()} (degree = {atom.GetDegree()})")

In [ ]:
# Loop over all atoms in the molecule
for atom in mol.GetAtoms():
    
    # Check whether the atom is part of any ring
    in_ring = atom.IsInRing() #atom.IsInRingSize(n) # True if the atom is in a ring of size n
    
    # Print the result in a readable way
    print(f"Atom {atom.GetIdx()} ({atom.GetSymbol()}) -> In ring: {in_ring}")

In [ ]:
# Loop directly over all bonds in the molecule
for bond in mol.GetBonds():
    
    # Get the indices of the bonded atoms
    begin_idx = bond.GetBeginAtomIdx()
    end_idx = bond.GetEndAtomIdx()
    
    # Get the atom symbols
    begin_symbol = mol.GetAtomWithIdx(begin_idx).GetSymbol()
    end_symbol = mol.GetAtomWithIdx(end_idx).GetSymbol()
    
    # Print bond information
    print(
        f"Bond: {begin_idx} ({begin_symbol}) "
        f"- {end_idx} ({end_symbol}) | "
        f"Type: {bond.GetBondType()}"
    )

In [ ]:
# Create a new molecule (toluene)
smiles_ring = "c1ccccc1C"
mol_ring = Chem.MolFromSmiles(smiles_ring)

# We need to add Hydrogens to the molecule to get the correct number of atoms
mol_ring = Chem.AddHs(mol_ring)

# Identify atoms that are part of a ring
ring_atoms = []

for atom in mol_ring.GetAtoms():
    if atom.IsInRing():
        ring_atoms.append(atom.GetIdx())

print("Ring atom indices:", ring_atoms)

# Draw the molecule highlighting ring atoms
img = Draw.MolToImage(mol_ring, highlightAtoms=ring_atoms)
display(img)

**Different SMILES represent the same molecule**

A SMILES string is a textual traversal of a molecular graph. If you change the starting atom or the order in which bonds are traversed, the string changes.

**What is a canonical SMILES and why is it important?**

A canonical SMILES is a unique, deterministic SMILES generated by an algorithm that assigns a fixed ordering to the atoms of a molecule.

This guarantees that the same molecule will always produce the same SMILES string.

Canonical SMILES are important because they:
- Avoid duplicates in chemical databases  
- Allow reliable molecule comparison  
- Provide a consistent identifier for storage and indexing  

In [ ]:
# Demonstrating that different SMILES represent the same molecular graph
smiles_list = ["CCO", "OCC", "C(O)C"]
mols = [Chem.MolFromSmiles(s) for s in smiles_list]

print("\nCanonical SMILES comparison:\n")
for smi in smiles_list:
    mol = Chem.MolFromSmiles(smi)
    canonical = Chem.MolToSmiles(mol, canonical=True)
    print(f"{smi} -> {canonical}")

# Display molecules side by side
Draw.MolsToGridImage(
    mols,
    legends=smiles_list,
    molsPerRow=3
)

## Exercises — Exploring `Mol` and Atom Objects

Create the following molecule:

```python
"CC(O)C=O"
```

Remember: a SMILES string is just text. How do we convert it into something RDKit can manipulate?

### Exercise 1 — Exploring the `Mol` Object

Use methods of the `Mol` object to answer the following:

1. How many atoms does the molecule have?
2. How many bonds does it have?
3. How many heavy atoms (non-hydrogen atoms)?
4. Does the molecule contain a ring?
5. What is the canonical SMILES of the molecule?

In [ ]:
# YOUR CODE HERE #

smiles = "CC(O)C=O"

# Create an RDKit molecule from this SMILES

### Exercise 2 — Looping Over Atoms

Loop over all atoms in the molecule and print the following information for each atom:

- Atom index
- Atom symbol
- Degree (number of directly bonded atoms)

In [ ]:
# YOUR CODE HERE #

# Molecular Descriptors and Properties

So far, we have explored molecular structure:
- Atoms  
- Bonds  
- Rings  
- Connectivity  

Now we move from structure to numbers.

A **molecular descriptor** is simply a numerical value calculated from a molecule. It converts chemical structure into quantitative information.

Descriptors allow us to:
- Compare molecules  
- Estimate properties  
- Build predictive models  

RDKit provides several types of descriptors:

- **Physical properties**  
  Examples: molecular weight, LogP (lipophilicity).

- **Topological descriptors**  
  Numbers derived from the molecular graph (connectivity and branching).

- **Connectivity indices**  
  Mathematical indices encoding how atoms are connected.

- **Count-based descriptors**  
  Simple counts such as number of atoms, bonds, rings, or heteroatoms.

These values transform molecular structure into features that can be analyzed computationally.

Let’s calculate some common descriptors and see what they tell us.

In [ ]:
from rdkit.Chem import Descriptors

# Total number of available descriptors
num_descriptors = len(Descriptors._descList)
print(f"Total number of RDKit descriptors: {num_descriptors}\n")

# Show the first 10 descriptor names
print("First 10 descriptors:")
for name, _ in Descriptors._descList[:10]:
    print("-", name)

In [ ]:
# SMILES string for propane
smiles = 'CCC' 
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)

# Call the descriptor directly
direct_value = Descriptors.MolWt(mol)
print("MolWt (direct call):", direct_value)

Example - Lipinski’s Rule of Five is a set of empirical guidelines used to evaluate whether a molecule is likely to exhibit good oral bioavailability.

In [ ]:
# SMILES string for propane
smiles = 'CCC' 
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)

# Molecular weight (average atomic masses, in g/mol)
MW = Descriptors.MolWt(mol)

# Number of nitrogen and oxygen atoms (NOT true H-bond acceptors)
HBA = Descriptors.NOCount(mol)

# Number of NH and OH groups (potential hydrogen bond donors)
HBD = Descriptors.NHOHCount(mol)

# Crippen LogP (estimated octanol/water partition coefficient, lipophilicity)
LogP = Descriptors.MolLogP(mol)


# Print descriptor values
print("Molecular Weight:", round(MW, 2))
print("HBA:", HBA)
print("HBD:", HBD)
print("LogP:", round(LogP, 2))


In [ ]:
# Apply Lipinski conditions
mw_ok = MW <= 500
hba_ok = HBA <= 10
hbd_ok = HBD <= 5
logp_ok = LogP <= 5

print("MW <= 500:", mw_ok)
print("HBA <= 10:", hba_ok)
print("HBD <= 5:", hbd_ok)
print("LogP <= 5:", logp_ok)


In [ ]:
# Final decision (at least 3 rules satisfied)
passed = (mw_ok + hba_ok + hbd_ok + logp_ok) >= 3
print("Passes Lipinski Rule of Five:", passed)



### Exercise — Exploring Available Descriptors

1. Choose any molecule of your interest.
2. Provide its SMILES representation.
3. Calculate LogP value for your molecule.

In [ ]:
# ============================
# YOUR CODE HERE
# ============================

# Create molecule from SMILES

# Calculate LogP (MolLogP) for the molecule

## From SMILES to 3D Structures

So far, we have worked with **SMILES strings** to represent molecules. SMILES allow us to describe molecular connectivity (which atoms are bonded to which). However, there is an important limitation:

SMILES strings describe molecules in **2D (topology only)**. They do not tell us how atoms are arranged in space.

In reality, molecules exist in **three dimensions**, and their 3D shape strongly influences:

- Reactivity  
- Molecular recognition  
- Docking interactions  
- Physical properties  
- Energy  

To work with realistic molecular geometries, we need to generate a **3D conformer**. The typical workflow to move from 2D to 3D is:

1. **Add hydrogens**  
   Geometry depends on all atoms, including hydrogens.

2. **Generate a 3D structure**  
   RDKit uses distance geometry to create a reasonable 3D arrangement of atoms.

3. **Optimize the geometry**  
   A force field (such as UFF or MMFF) adjusts the structure to reach a lower-energy conformation.

4. **Visualize the result**  
   We can then inspect the molecule in 3D.

In the following section, we will generate and visualize 3D conformers from SMILES using RDKit.

In [ ]:
# Create molecule from SMILES
mol = Chem.MolFromSmiles("COCC(C=O)CC(F)(F)CN")

# Show 2D depiction (no 3D coordinates yet)
display(mol)

# Add hydrogens (important for 3D geometry)
mol = Chem.AddHs(mol)

# Generate 3D coordinates (create one conformer)
AllChem.EmbedMolecule(mol, AllChem.ETKDG())

# Optimize geometry using a force field
AllChem.UFFOptimizeMolecule(mol)

In [ ]:
# Print atomic coordinates
conf = mol.GetConformer()
for atom in mol.GetAtoms():
    pos = conf.GetAtomPosition(atom.GetIdx())
    print(atom.GetSymbol(), pos.x, pos.y, pos.z)

In [ ]:
# Create a 3D viewer
view = py3Dmol.view(width=500, height=400)

# Convert RDKit molecule (with 3D coordinates) to MolBlock format
mol_block = Chem.MolToMolBlock(mol)

# Add the molecule to the viewer
view.addModel(mol_block, 'sdf')

# Set visualization style (stick model)
view.setStyle({'stick': {}})

# Adjust camera to fit the molecule
view.zoomTo()

# Display
view.show()

## Working with CSV Files containing SMILES

Unlike SDF files, a **CSV (Comma-Separated Values)** file does not store molecular structures directly.

Instead, it typically stores:

- SMILES strings (text representation of molecules)
- Molecular identifiers (names, codes)
- Calculated descriptors or other properties

A CSV file does **not** contain:
- 3D coordinates
- Explicit bond information

To work with molecules stored in a CSV file, we must:

1. Read the file as a table
2. Extract the SMILES column
3. Convert each SMILES string into an RDKit molecule object

Let’s load an example CSV file and inspect its contents.

In [ ]:
# Read CSV file
df = pd.read_csv("data/molecules.csv")
print("CSV content:")
print(df)

# Convert SMILES column to RDKit molecules
molecules = []
for smiles in df["SMILES"]:
    mol = Chem.MolFromSmiles(smiles)
    molecules.append(mol)

print(f"\nNumber of molecules created: {len(molecules)}\n")

------------------------------
## 🔎 Solutions

This section contains the suggested solutions for all exercises in the notebook.

Try to complete the exercises on your own before reviewing the answers. The goal is not only to obtain the correct output, but to understand the underlying concepts.

------------------------------

In [ ]:
# ================================
# SOLUTIONS — Block: Mol & Atoms
# ================================

from rdkit import Chem

# ----------------
# Exercise 1 — Mol object
# ----------------

mol = Chem.MolFromSmiles("CC(O)C=O")
mol_H = Chem.AddHs(mol)

print("Number of atoms:", mol_H.GetNumAtoms())
print("Number of bonds:", mol_H.GetNumBonds())
print("Number of heavy atoms:", mol_H.GetNumHeavyAtoms())
print("Number of rings:", mol_H.GetRingInfo().NumRings())
print("Canonical SMILES:", Chem.MolToSmiles(mol, canonical=True))

print("\n" + "="*50 + "\n")

# ----------------
# Exercise 2 — Loop over atoms
# ----------------

for atom in mol_H.GetAtoms():
    print(f"Atom index: {atom.GetIdx()}")
    print(f"Symbol: {atom.GetSymbol()}")
    print(f"Degree: {atom.GetDegree()}")
    print("-" * 40)

In [ ]:
# ==========================================
# SOLUTIONS — Block: Molecular Descriptors
# ==========================================

from rdkit import Chem
from rdkit.Chem import Descriptors

# ----------------
# Exercise 1 — Calculate LogP
# ----------------

mol = Chem.MolFromSmiles("CC(O)C=O")

logp = Descriptors.MolLogP(mol)

print("LogP (MolLogP):", round(logp, 3))